# Plot results for predicting hurricane track (distance) errors.
author: Elizabeth A. Barnes and Randal J. Barnes

In [1]:
import sys
sys.path.append('..')

import datetime
import importlib as imp
import os
import pickle
import pprint
import random
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

import tensorflow_probability as tfp
from sklearn import preprocessing
from silence_tensorflow import silence_tensorflow

from build_data import build_hurricane_data
import build_model
import experiment_settings
from save_model_run import save_model_run

from scipy.stats import multivariate_normal
import mahalanobis

mpl.rcParams["figure.facecolor"] = "white"
np.warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

In [2]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "05 August 2022"

silence_tensorflow()
tf.config.set_visible_devices([], 'GPU') # turn-off tensorflow-metal if it is on

DATA_PATH = "../data/"
MODEL_PATH = "saved_models/"

In [3]:
EXP_NAME_LIST = ("bivariate_normal_105_EPCP48",
                 "bivariate_normal_106_EPCP48",
                 "bivariate_normal_101_EPCP48",
                )

# Define functions

In [4]:
def get_error_metrics(y_pred_test, onehot_test):
    error_vec = np.zeros((y_pred_test.shape[0],2))
    for isample in np.arange(0,y_pred_test.shape[0]):
        mu_u,mu_v,sigma_u,sigma_v,rho = (y_pred_test[isample,0],
                                         y_pred_test[isample,1],
                                         y_pred_test[isample,2],
                                         y_pred_test[isample,3],
                                         y_pred_test[isample,4],
                                        )
        cov = np.array([[sigma_u**2, rho*sigma_v*sigma_u], [rho*sigma_v*sigma_u, sigma_v**2]])
        rv = multivariate_normal([mu_u, mu_v], cov)
        pred_x, pred_y = (rv.mean[0], rv.mean[1])
        error = [pred_x - onehot_test[isample,0], pred_y - onehot_test[isample,1]]
        error_vec[isample] = error

    rmse = np.sqrt(error_vec[:,0]**2 + error_vec[:,1]**2)
    rmse_cons = np.sqrt(onehot_test[:,0]**2 + onehot_test[:,1]**2)
    improvement = 100*(rmse.mean()-rmse_cons.mean())/rmse_cons.mean()

    print('    ' + str(improvement.round(2)) + '%') # negative implies improvement
    
    return rmse, rmse_cons, improvement

# Run metrics

In [5]:
for EXP_NAME in EXP_NAME_LIST:
    print('----' + EXP_NAME + '----')
    
    for TESTING_YEAR in np.arange(2013,2022):
        print(TESTING_YEAR)
        
        for RNG_SEED in (123, 234, 345):
            settings = experiment_settings.get_settings(EXP_NAME)

            network_seed = RNG_SEED
            settings['rng_seed'] = RNG_SEED
            settings["years_test"] = (TESTING_YEAR,)
            (
                data_summary,        
                x_train,
                onehot_train,
                x_val,
                onehot_val,
                x_test,
                onehot_test,        
                x_valtest,
                onehot_valtest,
                df_train,
                df_val,
                df_test,
                df_valtest,
            ) = build_hurricane_data(DATA_PATH, settings, verbose=0)

            # load the model
            model_name = (
                EXP_NAME + "_" + 
                str(TESTING_YEAR) + '_' +
                settings["uncertainty_type"] + '_' + 
                f"network_seed_{network_seed}_rng_seed_{settings['rng_seed']}"
            )
            try:
                model = tf.keras.models.load_model(MODEL_PATH + model_name + "_model", compile=False)
            except:
                break

            y_pred_test = model.predict(x_test)
            y_pred_valtest = model.predict(x_valtest)
            y_pred_train = model.predict(x_train)

            rmse, rmse_cons, improvement = get_error_metrics(y_pred_test, onehot_test)

----bivariate_normal_105_EPCP48----
2013
    7.26%
    1.43%
    1.65%
2014
    2.87%
    10.26%
    10.43%
2015
    1.43%
    5.99%
    4.18%
2016
    13.14%
    9.46%
    13.31%
2017
    4.64%
    9.35%
    4.43%
2018
    -0.93%
    -1.44%
    2.07%
2019
    6.36%
    0.91%
    1.89%
2020
    -8.88%
    -8.61%
    -7.03%
2021
    -11.71%
    -10.48%
    -9.08%
----bivariate_normal_106_EPCP48----
2013
    -3.2%
    -2.16%
    5.21%
2014
    -2.57%
    1.64%
    -3.33%
2015
    3.33%
    5.74%
    1.35%
2016
    6.05%
    9.75%
    5.1%
2017
    10.69%
    8.0%
    10.04%
2018
    -1.36%
    -3.03%
    -1.45%
2019
    -2.6%
2020
2021
----bivariate_normal_101_EPCP48----
2013
    0.88%
    1.71%
    3.98%
2014
    -1.43%
    -0.53%
    -1.85%
2015
    1.5%
    1.6%
    -0.22%
2016
    9.32%
    8.55%
    6.12%
2017
    7.41%
    4.94%
    10.99%
2018
    -2.07%
    -4.26%
    -3.72%
2019
    -0.92%
    -3.24%
    0.5%
2020
    -5.53%
    -4.78%
    -7.24%
2021
    -12.86%
    -14.36%
   